# External HU-density mask quality control

**Scientific purpose.** Apply the historical CT/mask geometry, HU-density, and connected-
component checks used to reject organ/background-mask failures and define the retained
103-case external cohort.

**Runnability classification:** runnable after notebook 09 using private patient-linked
NIfTI paths. **Inputs:** private C2 CT/mask paths. **Private outputs:** row-level QC and
exclusion manifests. Only the aggregate
retained count is public.


In [ ]:
from pathlib import Path
import sys


def locate_repository(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from within the cloned repository.")


REPO_ROOT = locate_repository()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from liver_tumor_pipeline.config import load_config, require_path

CONFIG_PATH = REPO_ROOT / "configs" / "paths.yaml"
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(
        "Create an untracked configs/paths.yaml from configs/paths.example.yaml and "
        "set the documented environment variables before running this workflow."
    )

CONFIG = load_config(CONFIG_PATH)
PRIVATE_ARTIFACT_ROOT = require_path(CONFIG, "private_artifact_root", must_exist=False)
OUTPUT_ROOT = require_path(CONFIG, "output_root", must_exist=False)

import numpy as np
import pandas as pd
import nibabel as nib
from scipy import ndimage

MASK_ROOT = PRIVATE_ARTIFACT_ROOT / "external" / "tumor_masks"
MANIFEST_PATH = MASK_ROOT / "external_manifest.csv"
QC_PATH = MASK_ROOT / "mask_density_qc.csv"
FILTERED_MANIFEST_PATH = MASK_ROOT / "external_manifest_filtered.csv"
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError("Private external tumor-mask manifest is unavailable; run notebook 09.")


## Density and geometry audit

C2 is used for this mask-type quality check. The thresholds are a cohort adapter QC rule,
not a tumor-biology classifier.


In [ ]:
def classify_density(mean_hu: float, p10_hu: float, p90_hu: float) -> str:
    if np.isfinite(p10_hu) and np.isfinite(p90_hu) and p10_hu < -300 and p90_hu > 30:
        return "bimodal_likely_misregistration"
    if not np.isfinite(mean_hu):
        return "invalid"
    if mean_hu < -200:
        return "likely_air_or_lung"
    if mean_hu < 0:
        return "suspect"
    if mean_hu <= 250:
        return "liver_range"
    return "suspect_high_density"


def audit_mask(row: pd.Series) -> dict:
    patient_key = str(row["patient_key"])
    ct_path = Path(str(row["ct_C2"]))
    mask_path = Path(str(row["mask_C2"]))
    result = {
        "patient_key": patient_key,
        "status": "invalid",
        "mean_hu": np.nan,
        "median_hu": np.nan,
        "p10_hu": np.nan,
        "p90_hu": np.nan,
        "mask_voxels": 0,
        "components": 0,
        "largest_component_fraction": np.nan,
    }
    if not ct_path.is_file() or not mask_path.is_file():
        result["status"] = "missing_input"
        return result
    ct = nib.load(str(ct_path)).get_fdata()
    mask = nib.load(str(mask_path)).get_fdata() > 0
    if ct.shape != mask.shape:
        result["status"] = "shape_mismatch"
        return result
    result["mask_voxels"] = int(mask.sum())
    if result["mask_voxels"] == 0:
        result["status"] = "empty_mask"
        return result
    values = ct[mask]
    values = values[np.isfinite(values)]
    if values.size == 0:
        return result
    result.update(
        mean_hu=float(values.mean()),
        median_hu=float(np.median(values)),
        p10_hu=float(np.percentile(values, 10)),
        p90_hu=float(np.percentile(values, 90)),
    )
    labeled, component_count = ndimage.label(mask)
    result["components"] = int(component_count)
    if component_count:
        sizes = np.bincount(labeled.ravel())[1:]
        result["largest_component_fraction"] = float(sizes.max() / mask.sum())
    result["status"] = classify_density(
        result["mean_hu"], result["p10_hu"], result["p90_hu"]
    )
    return result


In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
required_columns = {"patient_key", "ct_C2", "mask_C2"}
missing_columns = sorted(required_columns.difference(manifest.columns))
if missing_columns:
    raise ValueError(f"External manifest is missing required columns: {missing_columns}")

qc = pd.DataFrame([audit_mask(row) for _, row in manifest.iterrows()])
qc.to_csv(QC_PATH, index=False)
retained_keys = set(qc.loc[qc["status"] == "liver_range", "patient_key"].astype(str))
filtered = manifest[manifest["patient_key"].astype(str).isin(retained_keys)].copy()
filtered.to_csv(FILTERED_MANIFEST_PATH, index=False)
if len(filtered) != 103:
    raise RuntimeError(
        f"Mask-density QC retained {len(filtered)} cases; the locked external cohort contains 103."
    )
qc["status"].value_counts().rename_axis("status").reset_index(name="cases")


The row-level QC table contains patient-linked paths and remains private. Downstream
reporting may state only that 103 known-HCC cases were retained. This check does not make
the stress test a complete external validation cohort.
